In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, mean_squared_error, mean_absolute_error

X_train = pd.read_csv("X_train_prima.csv")
X_test = pd.read_csv("X_test_prima.csv")
y_train = pd.read_csv("y_train_prima.csv")
y_test = pd.read_csv("y_test_prima.csv")


In [ ]:
#skelearn y statsmodel esperan una serie no un df
y_train = y_train["Prima"] if "Prima" in y_train.columns else y_train.iloc[:, 0]
y_test = y_test["Prima"] if "Prima" in y_test.columns else y_test.iloc[:, 0]

In [ ]:
#las columnas despues de get dummys se guardan como bool en vez de numerico
X_train = X_train.astype(float)
X_test = X_test.astype(float)

# modelo con sklearn + evaluar en test

In [ ]:
modelo_sklearn = LinearRegression()
modelo_sklearn.fit(X=X_train, y=y_train)

In [ ]:
def data_frame_maker(model):
    coef_dict = {}
    coef_dict["intercepto"] = model.intercept_
    for i in range(1,len(model.feature_names_in_)):
        coef_dict[model.feature_names_in_[i]] = model.coef_[i]

    return pd.DataFrame(
        list(coef_dict.items()), columns=["Feature", "Coeficiente"]
    )

print(data_frame_maker(modelo))
print(f"Coeficiente de determinación R^2:", modelo.score(X_train_scaled, y_train))

In [ ]:
predicciones_sklearn = modelo_sklearn.predict(X=X_test)
mse = mean_squared_error(y_true=y_test, y_pred=predicciones_sklearn)
rmse = root_mean_squared_error(y_true=y_test, y_pred=predicciones_sklearn)
mae = mean_absolute_error(y_true=y_test, y_pred=predicciones_sklearn)

print(f"Primeras cinco predicciones: {predicciones_sklearn[0:5]}")
print(f"El error (mse) de test es: {round(mse, 2)}")
print(f"El error (rmse) de test es: {round(rmse, 2)}")
print(f"El error (mae) de test es: {round(mae, 2)}")

# modelo con statsmodels

In [ ]:
#resumen estadistico
X_train_sm = sm.add_constant(X_train, prepend=True)
modelo_sm = sm.OLS(endog=y_train, exog=X_train_sm)
modelo_sm = modelo_sm.fit()
print(modelo_sm.summary())

In [ ]:
#intervalos de confianza
intervalos_ci = modelo_sm.conf_int(alpha=0.05)
intervalos_ci.columns = ["2.5%", "97.5%"]
print(intervalos_ci)

# diagnostico de los residuos

In [ ]:
prediccion_train = modelo_sm.predict(exog=X_train_sm)
residuos_train = prediccion_train - y_train

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))

axes[0, 0].scatter(y_train, prediccion_train, edgecolors=(0, 0, 0), alpha=0.4)
axes[0, 0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()],
                "k--", color="black", lw=2)
axes[0, 0].set_title("Valor predicho vs valor real", fontsize=10, fontweight="bold")
axes[0, 0].set_xlabel("Real")
axes[0, 0].set_ylabel("Prediccion")

axes[0, 1].scatter(list(range(len(y_train))), residuos_train, edgecolors=(0, 0, 0), alpha=0.4)
axes[0, 1].axhline(y=0, linestyle="--", lw=2)
axes[0, 1].set_title("Residuos del modelo", fontsize=10, fontweight="bold")
axes[0, 1].set_xlabel("id")
axes[0, 1].set_ylabel("Residuo")

sns.histplot(data=residuos_train, stat="density", kde=True,
             line_kws={"linewidth": 1}, alpha=0.3, ax=axes[1, 0])
axes[1, 0].set_title("Distribucion de los residuos", fontsize=10, fontweight="bold")
axes[1, 0].set_xlabel("Residuo")

sm.qqplot(residuos_train, fit=True, line="q", ax=axes[1, 1], alpha=0.4, lw=2)
axes[1, 1].set_title("Q-Q residuos del modelo", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

# test de normalidad de los residuos

In [ ]:
shapiro_test = stats.shapiro(residuos_train)
print("Shapiro-Wilk:", shapiro_test)

k2, p_value = stats.normaltest(residuos_train)
print(f"D'Agostino K^2: estadistico = {k2}, p-value = {p_value}")

# predicciones

In [ ]:
predicciones_intervalo = modelo_sm.get_prediction(exog=X_train_sm).summary_frame(alpha=0.05)
print(predicciones_intervalo.head())